# Cellpose 3D Inference

This notebook runs 3D segmentation using a pre-trained Cellpose model.

The main steps are:
1.  **Configuration**: Set paths to the model, input image, and output file.
2.  **Model Loading**: Load the specified Cellpose model.
3.  **Inference**: Run the `eval` function on the 3D input image.
4.  **Save Output**: Save the resulting 3D mask as a `.tif` file.

In [1]:
import os
import numpy as np
import tifffile
import torch
from pathlib import Path
from cellpose import models
from scipy.ndimage import label as ndimage_label, generate_binary_structure
from skimage.morphology import remove_small_objects

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — change these two lines only
# ─────────────────────────────────────────────────────────────────────────────
MODEL_PATH   = "/work/scratch/rawat/models/cellpose3d/models/cellpose3d_calyx_All_v2"
INPUT_FOLDER = "/home/students/rawat/Final_dataset/all/test/images"
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_BASE            = "/home/students/rawat/Results"
VALID_XY_SIZES         = {1100, 1101}
BLOB_REMOVAL_THRESHOLD = 1450

XY_VOXEL_UM  = 0.0709
Z_VOXEL_UM   = 0.3
DIAMETER     = 30
CHANNEL_IDX  = 0
NUCLEUS_IDX  = 1

USE_GPU    = torch.cuda.is_available()
MODEL_NAME = Path(MODEL_PATH).name

print(f"Model name   : {MODEL_NAME}")
print(f"Model path   : {MODEL_PATH}")
print(f"Input folder : {INPUT_FOLDER}")
print(f"Output base  : {OUTPUT_BASE}/{MODEL_NAME}/")
print(f"GPU          : {USE_GPU}")



Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.4 
platform:       	linux 
python version: 	3.10.12 
torch version:  	2.7.1+cu118! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


Model name   : cellpose3d_calyx_All_v2
Model path   : /work/scratch/rawat/models/cellpose3d/models/cellpose3d_calyx_All_v2
Input folder : /home/students/rawat/Final_dataset/all/test/images
Output base  : /home/students/rawat/Results/cellpose3d_calyx_All_v2/
GPU          : True


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# PREPROCESSING HELPERS — copied from training_cellpose3d.py
# ─────────────────────────────────────────────────────────────────────────────

def fix_image_dims(img):
    if img.ndim == 2: return img[np.newaxis, ..., np.newaxis]
    if img.ndim == 3: return img[..., np.newaxis]
    if img.ndim == 4:
        if int(np.argmin(img.shape)) == 0:
            img = np.transpose(img, (1, 2, 3, 0))
    return img

def ensure_zyxc(img):
    """Converts (Z,C,Y,X) → (Z,Y,X,C) if needed."""
    if img.ndim == 4 and img.shape[1] <= 4 and img.shape[1] < img.shape[2]:
        img = np.transpose(img, (0, 2, 3, 1))
        print(f"  [fix] Transposed (Z,C,Y,X) → (Z,Y,X,C): {img.shape}")
    return img

def select_channels(img, channel_idx, nucleus_idx):
    if img.ndim == 3: return img[..., np.newaxis]
    cyto = img[..., [channel_idx]]
    if nucleus_idx is not None and nucleus_idx < img.shape[-1]:
        return np.concatenate([cyto, img[..., [nucleus_idx]]], axis=-1)
    return cyto

def normalize_slice(sl):
    sl = sl.astype(np.float32)
    for c in range(sl.shape[-1]):
        lo, hi = np.percentile(sl[..., c], (0.00001, 99.99999))
        sl[..., c] = (sl[..., c] - lo) / (hi - lo) if hi > lo else 0.0
    return np.clip(sl, 0.0, 1.0)

def preprocess_volume(path):
    """Load and preprocess a TIFF the same way as training."""
    vol = tifffile.imread(str(path))
    vol = ensure_zyxc(fix_image_dims(vol))                         # → (Z,Y,X,C)
    vol = select_channels(vol, CHANNEL_IDX, NUCLEUS_IDX)           # → (Z,Y,X,2)
    for z in range(vol.shape[0]):
        vol[z] = normalize_slice(vol[z])                           # per-slice norm
    return vol.astype(np.float32)

print("Helper functions ready.")

Helper functions ready.


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# SCAN FOLDER FOR VALID IMAGES
# ─────────────────────────────────────────────────────────────────────────────
all_tifs = sorted(Path(INPUT_FOLDER).glob("*.tif"))
valid_images = []

print(f"Scanning: {INPUT_FOLDER}\n")
for p in all_tifs:
    arr = tifffile.imread(str(p))
    y, x = arr.shape[-2], arr.shape[-1]
    if y in VALID_XY_SIZES and x in VALID_XY_SIZES:
        valid_images.append(p)
        print(f"  VALID   {p.name}  {arr.shape}")
    else:
        print(f"  SKIP    {p.name}  {arr.shape}  (XY={y}x{x})")

print(f"\n{len(valid_images)} image(s) will be processed.")

Scanning: /home/students/rawat/Final_dataset/all/test/images

  SKIP    Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL.tif  (70, 2, 1834, 1834)  (XY=1834x1834)
  VALID   Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL_1100.tif  (70, 2, 1100, 1100)
  VALID   PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit.ome_deconv.tif  (2, 76, 1101, 1101)
  VALID   PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit.tif  (2, 76, 1101, 1101)
  VALID   PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit_dog.tif  (76, 2, 1101, 1101)

4 image(s) will be processed.


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# LOAD MODEL ONCE
# ─────────────────────────────────────────────────────────────────────────────
print(f"Loading model: {MODEL_NAME} ...")
model = models.CellposeModel(gpu=USE_GPU, pretrained_model=MODEL_PATH)
print("Model ready.")

pretrained model /home/students/rawat/.cellpose/models/cpsam not found, using default model


Loading model: cellpose3d_calyx_All_v2 ...
Model ready.


In [5]:
from tqdm import tqdm

# ─────────────────────────────────────────────────────────────────────────────
# INFERENCE LOOP — all valid images
# ─────────────────────────────────────────────────────────────────────────────
for img_path in valid_images:
    print(f"\n{'─'*60}")
    print(f"Processing: {img_path.name}")

    vol = preprocess_volume(img_path)   # (Z, Y, X, 2)
    print(f"  Preprocessed: {vol.shape}  (Z,Y,X,C)")

    masks, flows, _ = model.eval(
        vol,
        do_3D=True,
        z_axis=0,
        channel_axis=3,
        anisotropy=Z_VOXEL_UM / XY_VOXEL_UM,
        diameter=DIAMETER,
        normalize=False,
        batch_size=1,
    )

    # ── Blob removal ──────────────────────────────────────────────────────────
    clean   = remove_small_objects(masks.astype(bool), min_size=BLOB_REMOVAL_THRESHOLD, connectivity=1)
    # re-label after removal
    from scipy.ndimage import label as ndimage_label, generate_binary_structure
    clean, n_final = ndimage_label(clean, structure=generate_binary_structure(3, 3))
    print(f"  Objects found      : {len(np.unique(masks)) - 1}")
    print(f"  After blob removal : {n_final} objects  (threshold={BLOB_REMOVAL_THRESHOLD} vox)")

    # ── Save ──────────────────────────────────────────────────────────────────
    out_path = Path(OUTPUT_BASE) / MODEL_NAME / f"{img_path.stem}_mask.tif"
    os.makedirs(out_path.parent, exist_ok=True)
    dtype = np.uint32 if n_final > 65535 else np.uint16
    tifffile.imwrite(str(out_path), clean.astype(dtype))
    print(f"  Saved → {out_path}")

print(f"\n{'='*60}")
print(f"Done. Processed {len(valid_images)} image(s).")


────────────────────────────────────────────────────────────
Processing: Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL_1100.tif
  [fix] Transposed (Z,C,Y,X) → (Z,Y,X,C): (70, 1100, 1100, 2)
  Preprocessed: (70, 1100, 1100, 2)  (Z,Y,X,C)


Resizing is depricated in v4.0.1+


  Objects found      : 1685
  After blob removal : 28 objects  (threshold=1450 vox)
  Saved → /home/students/rawat/Results/cellpose3d_calyx_All_v2/Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL_1100_mask.tif

────────────────────────────────────────────────────────────
Processing: PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit.ome_deconv.tif
  Preprocessed: (76, 1101, 1101, 2)  (Z,Y,X,C)


Resizing is depricated in v4.0.1+


  Objects found      : 4105
  After blob removal : 50 objects  (threshold=1450 vox)
  Saved → /home/students/rawat/Results/cellpose3d_calyx_All_v2/PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit.ome_deconv_mask.tif

────────────────────────────────────────────────────────────
Processing: PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit.tif
  Preprocessed: (76, 1101, 1101, 2)  (Z,Y,X,C)


Resizing is depricated in v4.0.1+


  Objects found      : 0
  After blob removal : 0 objects  (threshold=1450 vox)
  Saved → /home/students/rawat/Results/cellpose3d_calyx_All_v2/PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit_mask.tif

────────────────────────────────────────────────────────────
Processing: PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit_dog.tif
  [fix] Transposed (Z,C,Y,X) → (Z,Y,X,C): (76, 1101, 1101, 2)
  Preprocessed: (76, 1101, 1101, 2)  (Z,Y,X,C)


Resizing is depricated in v4.0.1+


  Objects found      : 4213
  After blob removal : 50 objects  (threshold=1450 vox)
  Saved → /home/students/rawat/Results/cellpose3d_calyx_All_v2/PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit_dog_mask.tif

Done. Processed 4 image(s).
